In [1]:
import csv

# read demand data
f = open("module2_quiz_P2a.csv")
csvfile = csv.DictReader(f)
headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

Groups = []
Demand = {}
Months = range(1,6)
for row in table:
    Groups.append(row['Demand (tonne)'])
    for m in Months:
        Demand[(row['Demand (tonne)'], m)] = float(row[str("Month "+str(m))])

# read cost data
f = open("module2_quiz_P2b.csv")
csvfile = csv.DictReader(f)
headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()
                
Cost = {}
for row in table:
    for m in Months:
        Cost[(row['Production cost ($/tonne)'], m)] = float(row[str("Month "+str(m))])

In [2]:
ProcessingCapacity = 175
HoldingCost = 50
StorageCapacity = 50

In [3]:
Groups

['Large', 'Medium', 'Small']

In [4]:
Months

range(1, 6)

In [5]:
list(range(1, 6))

[1, 2, 3, 4, 5]

In [6]:
Demand

{('Large', 1): 31.0,
 ('Large', 2): 25.0,
 ('Large', 3): 45.0,
 ('Large', 4): 59.0,
 ('Large', 5): 44.0,
 ('Medium', 1): 50.0,
 ('Medium', 2): 41.0,
 ('Medium', 3): 70.0,
 ('Medium', 4): 80.0,
 ('Medium', 5): 60.0,
 ('Small', 1): 42.0,
 ('Small', 2): 30.0,
 ('Small', 3): 55.0,
 ('Small', 4): 72.0,
 ('Small', 5): 56.0}

In [7]:
Cost

{('Large', 1): 286.0,
 ('Large', 2): 273.0,
 ('Large', 3): 305.0,
 ('Large', 4): 337.0,
 ('Large', 5): 353.0,
 ('Medium', 1): 304.0,
 ('Medium', 2): 290.0,
 ('Medium', 3): 324.0,
 ('Medium', 4): 359.0,
 ('Medium', 5): 375.0,
 ('Small', 1): 313.0,
 ('Small', 2): 299.0,
 ('Small', 3): 334.0,
 ('Small', 4): 370.0,
 ('Small', 5): 386.0}

In [39]:
from docplex.mp.model import Model
mdl = Model()

In [40]:
# variables
Processed = mdl.continuous_var_matrix(Groups, Months, lb=0, name='Processed')

In [41]:
# auxiliary variables: inventory at end of month
Inventory = mdl.continuous_var_dict(Months, lb=0, ub=StorageCapacity, name='Inventory')

In [42]:
# objective: minimize cost
TotalProcessingCost = mdl.sum(Processed[(g,m)] * Cost[(g,m)] for g in Groups for m in Months)
TotalHoldingCost = mdl.sum(HoldingCost*Inventory[m] for m in Months)
mdl.minimize(TotalProcessingCost + TotalHoldingCost)

mdl.add_kpi(TotalProcessingCost, 'Total processing cost')
mdl.add_kpi(TotalHoldingCost, 'Total holding cost')

DecisionKPI(name=Total holding cost,expr=50Inventory_1+50Inventory_2+50Inventory_3+50Inventory_4+50Invent..)

In [43]:
# total processing per month constraint - at most 175 tonnes of potatoes/month
for m in Months:
    mdl.add_constraint(mdl.sum(Processed[(g,m)] for g in Groups) <= ProcessingCapacity)

In [45]:
# demand per month per group constraint
mdl.add_constraint(mdl.sum(Processed[(g,m)] for g in Groups for m in Months) >= mdl.sum(Demand[(g,m)] for g in Groups for m in Months))
             

docplex.mp.LinearConstraint[](Processed_Large_1+Processed_Large_2+Processed_Large_3+Processed_Large_4+Processed_Large_5+Processed_Medium_1+Processed_Medium_2+Processed_Medium_3+Processed_Medium_4+Processed_Medium_5+Processed_Small_1+Processed_Small_2+Processed_Small_3+Processed_Small_4+Processed_Small_5,GE,760)

In [ ]:
# inventory balance constraints
for m in Months:
    if m==1:
        mdl.add_constraint(Inventory[1] == Produced[1] - Demand[1])
    else:
        mdl.add_constraint(Inventory[m] == Inventory[m-1] + Produced[m] - Demand[m])

In [16]:
# inventory balance constraints

#Inventory[('Large', 0)] = 0
Inventory[0] = 5.3
#Inventory[('Small', 0)] = 0

for g in Groups:
    if g == 'Medium':
        mdl.add_constraint(mdl.sum(Inventory[m] for m in Months) \
                   == mdl.sum(Inventory[m-1] for m in Months)*0.96 \
                   + mdl.sum(Processed[(g,m)] for g in Groups for m in Months) \
                   - mdl.sum(Demand[(g,m)] for g in Groups for m in Months))  
                                                       
                                                                                                                          

docplex.mp.LinearConstraint[](Inventory_Large_1+Inventory_Large_2+Inventory_Large_3+Inventory_Large_4+Inventory_Large_5+Inventory_Medium_1+Inventory_Medium_2+Inventory_Medium_3+Inventory_Medium_4+Inventory_Medium_5+Inventory_Small_1+Inventory_Small_2+Inventory_Small_3+Inventory_Small_4+Inventory_Small_5,EQ,Processed_Large_1+Processed_Large_2+Processed_Large_3+Processed_Large_4+Processed_Large_5+Processed_Medium_1+Processed_Medium_2+Processed_Medium_3+Processed_Medium_4+Processed_Medium_5+Processed_Small_1+Processed_Small_2+Processed_Small_3+Processed_Small_4+Processed_Small_5+0.960Inventory_Large_1+0.960Inventory_Large_2+0.960Inventory_Large_3+0.960Inventory_Large_4+0.960Inventory_Medium_1+0.960Inventory_Medium_2+0.960Inventory_Medium_3+0.960Inventory_Medium_4+0.960Inventory_Small_1+0.960Inventory_Small_2+0.960Inventory_Small_3+0.960Inventory_Small_4-754.912)

In [17]:
# solve 
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.00193501,status='optimal')

In [18]:
mdl.print_solution()

objective: 231609.400
status: OPTIMAL_SOLUTION(2)
  Processed_Large_1=175.000
  Processed_Large_2=175.000
  Processed_Large_3=175.000
  Processed_Large_4=175.000
  Processed_Large_5=60.000
  Inventory_Small_5=5.088


In [19]:
mdl.report()

* model docplex_model1 solved with objective = 231609.400
*  KPI: Total production cost = 231355.000
*  KPI: Total holding cost    = 254.400
